### Drive access + dataset paths

Colab inside the VS Code extension cannot show the UI required by `google.colab.drive`. The next cell auto-detects whether we're in Colab or a local VS Code runtime and either mounts Drive (Colab) or mirrors the Drive folder locally via the Google Drive API. To run locally:

1. Place `SnowPole_Detection_Dataset/` next to this notebook **or** set `SNOWPOLE_DATASET_ROOT` to its location.
2. If you need to download from Drive instead, create a Google Cloud OAuth client (`Desktop` type), save the JSON as `client_secret.json` next to this notebook (or point `GOOGLE_DRIVE_CLIENT_SECRET` to it), and export `SNOWPOLE_DATASET_FOLDER_ID` with the shared folder ID.
3. (Optional) Override where the mirror is stored by setting `SNOWPOLE_DATASET_ROOT`.

After this setup the rest of the notebook can rely on the unified `DATASET_ROOT` regardless of environment.

In [10]:
os.getcwd()

COMB_ROOT  = Path("SnowPole_Detection_Dataset/images")

# 1-channel range-normalized images
RANGE_ROOT = Path("SnowPole_Detection_Dataset/range-normalized-continuous")

# New 4-channel dual-input dataset
DUAL_ROOT  = Path("SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range")
DUAL_ROOT.mkdir(parents=True, exist_ok=True)

print("COMB_ROOT :", COMB_ROOT)
print("RANGE_ROOT:", RANGE_ROOT)
print("DUAL_ROOT :", DUAL_ROOT)

COMB_ROOT : SnowPole_Detection_Dataset/images
RANGE_ROOT: SnowPole_Detection_Dataset/range-normalized-continuous
DUAL_ROOT : SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range


: 

### Branch integration blueprint (pseudo-color + normalized range)

![Pseudo-color and normalized-range fusion](attachment:image.png)

The sketch maps directly onto the cells below:

1. **Branch A – pseudo-color RGB (Near-IR/Signal/Reflectivity).**
   - Source: the `SnowPole_Detection_Dataset/Combination4_range_signal_reflec` images already contain three modalities rendered to RGB. Keep them as-is (or re-map channels if you prefer NIR/Signal/Reflect). This branch feeds the existing `COMB_ROOT` pipeline (`make_dual_split_npy`).
   - Implementation hook: right before stacking in `make_dual_split_npy`, pass the 3-channel tensor through a lightweight encoder (two Conv–BN–SiLU blocks) to normalize statistics prior to fusion.

2. **Branch B – normalized range depth.**
   - Source: continuous tensors produced by `scripts/normalize_range_images_continuous.py` (log scaling to [0,1] with the 80 m cap). Replace the current uint8 channel with this float map to preserve depth gradients.
   - Optional geocentric embedding (HHA): add height-above-ground `H`, horizontal disparity `D`, and angle wrt gravity `A` by combining LiDAR calibration + range map. This produces a 3-channel tensor before the branch B lightweight encoder so both branches output matching dimensions for fusion.

3. **Mid-fusion (CNX block).**
   - After both lightweight encoders, concatenate outputs and insert an attention/gating block: e.g., `CrossModalConv` = depthwise separable 3×3 conv + cross-attention (query from pseudo-color, key/value from range). This can live right before the `DUAL_ROOT` export or inside a custom Ultralytics model YAML (replace the first `Conv` block with two parallel stems plus a fusion module).

4. **Tweak layer prior to YOLO backbone.**
   - The diagram’s “TWEAK LAYER” corresponds to a 1×1 conv (channel mixer) followed by depthwise 3×3 conv. Insert this as the very first block in `yolov9t_dual.yaml` so the fused feature map matches the standard YOLO stem dimensions (e.g., 64 channels) before flowing through the rest of the network.

5. **Where to ingest in this notebook.**
   - **Data prep cell (`make_dual_split_npy`)**: split logic into two loaders (pseudo-color branch + normalized range branch). Save both intermediate tensors if you want to debug per-branch statistics.
   - **Model YAML cell**: duplicate the first `[Conv]` block so you have two parallel branches, then inject a `Concat` + `Conv` that implements the CNX/tweak layer shown above.
   - **Training cell**: keep the same `yolo train ...` command, but point `model=` to the updated YAML/weights so the Ultralytics trainer instantiates both branches.

> **Tip:** keep the branch encoders lightweight (depthwise or MobileNet-style) to avoid blowing up FLOPs. The continuous 80 m range signals carry most of the depth cues, so prioritize preserving their dynamic range rather than heavy convolution early on.

## Dual-branch experiment roadmap (RGB + continuous 80 m range)

**idea.** Stress-test YOLOv9t with the 4-channel fusion tensor produced by the `range_log_norm_0_1()` pipeline (LiDAR range → log scale → [0, 1] float, capped at 80 m) and quantify how alternative regression/objectness losses affect skewed pole detections.


### Loss-function menu & when to use it

| Component | Default | Recommended alternatives | Why it helps (esp. skewed poles) |
|-----------|---------|--------------------------|----------------------------------|
| Box regression | CIoU (Ultralytics default) | Smooth L1 / Huber (LGMMFusion, 2024), GIoU/DIoU, SIoU | Smooth L1 stabilizes gradients when the new range channel sharpens depth edges; SIoU penalizes angular misalignment, useful when poles are tall/slender. |
| Classification/objectness | BCE + focal term | Quality/Generalized Focal Loss (QFL/GFL) or Varifocal Loss | Jointly encodes class score + IoU so false positives from the extra channel are down-weighted (LearnOpenCV, 2024). |
| Center heatmap/objectness | Standard BCE | IoU-weighted Gaussian heatmap (TransFusion-L/LGMMFusion) | Encourages better center localization when labels are sparse or skewed. |
| Auxiliary depth consistency (optional) | – | L1 between predicted object radial depth and range channel mean | Regularizes the range branch so 80 m normalization stays calibrated. |

#### What each component actually does
- **Box regression losses (CIoU vs Smooth L1/GIoU/SIoU).** These supervise the offset predictions for each bounding box. Smooth L1 clips large residuals, making convergence smoother when the depth channel introduces sudden gradients. GIoU/DIoU/SIoU add terms for overlap, center distance, and orientation, which is critical because poles are tall and easily misaligned; SIoU explicitly penalizes mismatch in angle/shape to keep elongated boxes upright.
- **Classification/objectness losses (BCE vs QFL/VFL).** The default BCE treats class presence independently from box quality. QFL/VFL replace the binary label with the target IoU, so the confidence score already reflects localization quality. That prevents the extra LiDAR channel from inflating logits on poorly-localized poles and reduces skewed-object false positives.
- **Center/heatmap loss.** A Gaussian heatmap supervision tells the model where object centers should be. Adding IoU weighting (as in TransFusion-L) pushes the network to focus on well-overlapping predictions, improving recall on thin poles whose centroids are easy to miss with plain BCE.
- **Auxiliary depth consistency.** Because Branch B is a continuous [0,1] representation of 0–80 m, an extra L1 between the predicted object depth (or anchor z-proxy) and the range map mean keeps the LiDAR channel physically meaningful, preventing drift when augmentations mix RGB+range statistics.

### 3. Suggested experiment matrix

1. **Baseline RGB** – run stock YOLOv9t (3 ch, default losses) → reference mAP.
2. **Dual naive** – 4-channel input, default losses → measure lift.
3. **Dual + Smooth L1** – set `ultralytics.yolo.cfg.default.box = "smooth_l1"` or patch trainer to swap `Loss.box`. Track whether angular/elongated poles improve.
4. **Dual + SIoU** – enable SIoU (`train box=2.5` and `siou=True` in recent Ultralytics builds) to penalize shape/orientation mismatch.
5. **Dual + QFL/VFL** – `yolo train ... loss=varifocal` or integrate [GFL head](https://learnopencv.com/yolo-loss-function-gfl-vfl-loss/) to correlate confidence with localization (helps skewed objects, often under-labeled).
6. **Dual + heatmap** – plug in LGMMFusion-style IoU-weighted gaussian head; monitor center heatmap MAE.
7. **Dual + depth consistency** – add auxiliary L1 on continuous range stats vs predicted 3D center distance (encourage monotonicity up to 80 m).

#### Why these experiments help the dual network
- **Baseline RGB vs Dual naive:** quantifies the raw benefit of adding the normalized range channel so you know whether later tweaks are worth the added complexity.
- **Smooth L1:** mitigates exploding gradients that come from sharp range discontinuities (e.g., near ground-truth pole edges). Expect smoother training curves and fewer NaNs when the dataset includes bright depth spikes.
- **SIoU:** explicitly models size/orientation, so the network stops predicting squashed boxes for tall poles. Improves AP for skewed/tilted poles where overlap alone is insufficient.
- **QFL/VFL:** aligns classification confidence with localization quality, crucial when the LiDAR branch boosts activations even for partially visible poles. Reduces false alarms in cluttered scenes.
- **Heatmap head:** improves object centers by supervising them with a Gaussian target weighted by IoU—vital for poles that occupy only a few pixels horizontally.
- **Depth consistency:** enforces that predicted poles respect the 0–80 m normalization, reducing mismatches between Branch A (appearance) and Branch B (range). Helps generalization to new lighting since the depth channel becomes a trusted signal.

### 4. Practical setup notes

1. **Dataset integrity** – ensure `SnowPole_Detection_Dataset/range-normalized-continuous/{train,valid,test}` exists (script now respects `SNOWPOLE_DATASET_ROOT`).
2. **4-channel loaders** – confirm augmentations keep the range channel untouched (disable `hsv_*` on channel 4).
3. **Loss toggles** – Ultralytics `yolo` CLI allows overriding `box`, `cls`, and `dfl`. For custom QFL/SIoU heads, clone `ultralytics/nn/tasks.py` into the repo and register your loss (see LGMMFusion §Detection head and loss function).
4. **Skewed-object splits** – create a metadata CSV capturing pole tilt/height; stratify validation so skewed cases are represented when comparing losses.

> **Key references:** LGMMFusion (Cheng et al., *PLOS ONE* 2024) for Smooth L1 + IoU heatmaps in LiDAR-camera fusion; GFL/VFL (Li et al., 2020-2021) for quality-aware focal losses; TransFusion-L (Chen et al., 2022) for cross-modal attention + gaussian center supervision.


In [ ]:
import os
import sys
import io
import json
import subprocess
import importlib
from pathlib import Path
from typing import Optional

NOTEBOOK_DIR = Path().resolve()
IN_COLAB = "google.colab" in sys.modules


def _resolve_local_dataset_root() -> Path:
    env_root = os.environ.get("SNOWPOLE_DATASET_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()
    candidates = [
        NOTEBOOK_DIR / "SnowPole_Detection_Dataset",
        NOTEBOOK_DIR / "data" / "SnowPole_Detection_Dataset",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return candidates[0].resolve()


if IN_COLAB:
    from google.colab import drive  # type: ignore

    drive.mount('/content/drive', force_remount=True)
    DATASET_ROOT = Path('/content/drive/MyDrive/SnowPole_Detection_Dataset').resolve()
else:
    DATASET_ROOT = _resolve_local_dataset_root()

    def _ensure_packages():
        missing = []
        for module_name, pip_name in [
            ("googleapiclient.discovery", "google-api-python-client"),
            ("google_auth_oauthlib.flow", "google-auth-oauthlib"),
            ("google.auth.transport.requests", "google-auth"),
        ]:
            try:
                importlib.import_module(module_name)
            except ImportError:
                missing.append(pip_name)
        if missing:
            subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

    _ensure_packages()
    from googleapiclient.discovery import build  # type: ignore
    from googleapiclient.http import MediaIoBaseDownload  # type: ignore
    from google.oauth2.credentials import Credentials  # type: ignore
    from google_auth_oauthlib.flow import InstalledAppFlow  # type: ignore
    from google.auth.transport.requests import Request  # type: ignore

    SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]

    def _resolve_client_secret() -> Path:
        candidates = [
            os.environ.get("GOOGLE_DRIVE_CLIENT_SECRET"),
            NOTEBOOK_DIR / "client_secret.json",
        ]
        for candidate in candidates:
            if not candidate:
                continue
            path = Path(candidate).expanduser()
            if path.exists():
                return path
        raise FileNotFoundError(
            "Provide a Google OAuth client secret JSON via GOOGLE_DRIVE_CLIENT_SECRET or client_secret.json next to the notebook"
        )

    def _get_drive_service():
        client_secret = _resolve_client_secret()
        token_path = NOTEBOOK_DIR / ".cache" / "drive_token.json"
        creds: Optional[Credentials] = None
        if token_path.exists():
            creds = Credentials.from_authorized_user_file(str(token_path), SCOPES)
        if not creds or not creds.valid:
            if creds and creds.expired and creds.refresh_token:
                creds.refresh(Request())
            else:
                flow = InstalledAppFlow.from_client_secrets_file(str(client_secret), SCOPES)
                creds = flow.run_local_server(port=0, prompt='consent')
            token_path.parent.mkdir(parents=True, exist_ok=True)
            token_path.write_text(creds.to_json())
        return build('drive', 'v3', credentials=creds, cache_discovery=False)

    def _download_file(service, file_info, destination: Path):
        destination.parent.mkdir(parents=True, exist_ok=True)
        expected_size = int(file_info.get('size', 0)) if file_info.get('size') else None
        if destination.exists() and expected_size and destination.stat().st_size == expected_size:
            return
        request = service.files().get_media(fileId=file_info['id'])
        with open(destination, 'wb') as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                _, done = downloader.next_chunk()
        if expected_size and destination.stat().st_size != expected_size:
            raise IOError(f"Incomplete download for {destination}")

    def _sync_folder(service, folder_id: str, destination: Path):
        destination.mkdir(parents=True, exist_ok=True)
        page_token = None
        while True:
            response = service.files().list(
                q=f"'{folder_id}' in parents and trashed = false",
                fields="nextPageToken, files(id, name, mimeType, size)",
                pageToken=page_token,
            ).execute()
            for item in response.get('files', []):
                target = destination / item['name']
                if item['mimeType'] == 'application/vnd.google-apps.folder':
                    _sync_folder(service, item['id'], target)
                else:
                    _download_file(service, item, target)
            page_token = response.get('nextPageToken')
            if not page_token:
                break

    def ensure_dataset_local(local_root: Path, folder_id: str):
        service = _get_drive_service()
        _sync_folder(service, folder_id, local_root)
        print(f"Google Drive folder {folder_id} mirrored to {local_root}")

    folder_id = os.environ.get('SNOWPOLE_DATASET_FOLDER_ID')
    dataset_missing = not DATASET_ROOT.exists() or not any(DATASET_ROOT.iterdir())
    if dataset_missing:
        if folder_id:
            ensure_dataset_local(DATASET_ROOT, folder_id)
        else:
            raise FileNotFoundError(
                "SnowPole_Detection_Dataset not found locally. Either unzip the dataset under "
                f"{DATASET_ROOT} or set SNOWPOLE_DATASET_ROOT/SNOWPOLE_DATASET_FOLDER_ID to mirror it from Drive."
            )

print(f"DATASET_ROOT set to {DATASET_ROOT}")

In [20]:
# !yolo train model=yolov9t.pt epochs=150 imgsz=1024 device=0 batch=2 data=/content/drive/MyDrive/data.yaml project=/content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec

: 

In [21]:
# !yolo val \
#   model=/content/drive/.shortcut-targets-by-id/1ofcfwBISHeUE4V1x6Bj-J5Mw3-QSUHFB/SnowPole_Detection_Dataset/comb4-range-signal-reflec-v11n/train2/weights/best.pt \
#   data=/content/drive/MyDrive/data.yaml \
#   split=test \
#   imgsz=1024 \
#   device=0 \
#   batch=16 \
#   project="comb5_signal_reflec_range_11n" \
#   name="comb5_signal_reflec_range_11n_test_eval"


: 

In [11]:
import shutil
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm

# Ensure these are set somewhere above in your notebook:
# COMB_ROOT = Path(".../comb_dataset_root")   # contains images/ and labels/
# RANGE_ROOT = Path(".../range-normalized-continuous")  # contains train/valid/test .npy files
# DUAL_ROOT  = Path(".../dual_dataset_root")  # target root where we will write images/ and labels/

def make_dual_split_npy(
    split: str,
    comb_root: Path,
    range_root: Path,
    dual_root: Path,
    save_as_png: bool = True,
):
    """Create dual 4-channel tensors (RGB + range) and store both PNG and .npy copies."""
    comb_img_dir = comb_root / split
    src_lbl_dir = comb_root.parent / "labels" / split
    range_npy_dir = range_root / split
    dual_img_dir = dual_root / "images" / split
    dual_lbl_dir = dual_root / "labels" / split

    dual_img_dir.mkdir(parents=True, exist_ok=True)
    dual_lbl_dir.mkdir(parents=True, exist_ok=True)

    if src_lbl_dir.exists():
        for lbl in src_lbl_dir.glob("*.txt"):
            shutil.copy2(lbl, dual_lbl_dir / lbl.name)
    else:
        print(f"Warning: source label dir not found: {src_lbl_dir}")

    img_files = sorted(comb_img_dir.glob("*.*"))
    print(f"[{split}] comb images found: {len(img_files)}")

    png_written = 0
    npy_written = 0

    for comb_path in tqdm(img_files, desc=f"make_dual_split ({split})"):
        stem = comb_path.stem

        comb = cv2.imread(str(comb_path), cv2.IMREAD_COLOR)
        if comb is None:
            print("Could not read comb image:", comb_path)
            continue

        range_npy_path = range_npy_dir / f"{stem}.npy"
        if not range_npy_path.exists():
            print(f"Missing range .npy for {stem} -> {range_npy_path} (skipping)")
            continue

        try:
            range_arr = np.load(str(range_npy_path))
        except Exception as e:
            print(f"Failed to load {range_npy_path}: {e}")
            continue

        if range_arr.ndim == 3 and range_arr.shape[0] in (1,):
            range_arr = np.squeeze(range_arr, axis=0)
        if range_arr.ndim == 3 and range_arr.shape[2] == 1:
            range_arr = np.squeeze(range_arr, axis=2)
        if range_arr.ndim != 2:
            print(f"Unexpected shape for range npy {range_npy_path}: {range_arr.shape} (skipping)")
            continue

        range_arr = np.clip(range_arr.astype(np.float32), 0.0, 1.0)
        range_uint8 = (range_arr * 255.0).astype(np.uint8)

        if range_uint8.shape != comb.shape[:2]:
            range_uint8 = cv2.resize(range_uint8, (comb.shape[1], comb.shape[0]), interpolation=cv2.INTER_NEAREST)

        rgba_uint8 = np.dstack([comb, range_uint8])  # (H, W, 4)

        if save_as_png:
            out_img_path = dual_img_dir / f"{stem}.png"
            cv2.imwrite(str(out_img_path), rgba_uint8)
            png_written += 1

        out_npy_path = dual_img_dir / f"{stem}.npy"
        np.save(out_npy_path, rgba_uint8, allow_pickle=False)
        npy_written += 1

    print(
        f"[{split}] done. Dual images in {dual_img_dir} | labels in {dual_lbl_dir} | "
        f"{png_written} PNG updated, {npy_written} NPY updated"
    )


for split in ["train", "valid", "test"]:
    make_dual_split_npy(split, comb_root=COMB_ROOT, range_root=RANGE_ROOT, dual_root=DUAL_ROOT, save_as_png=True)


[train] comb images found: 0


make_dual_split (train): 0it [00:00, ?it/s]


[train] done. Dual images written to SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/images/train, labels copied to SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/labels/train
[valid] comb images found: 0


make_dual_split (valid): 0it [00:00, ?it/s]


[valid] done. Dual images written to SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/images/valid, labels copied to SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/labels/valid
[test] comb images found: 0


make_dual_split (test): 0it [00:00, ?it/s]

[test] done. Dual images written to SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/images/test, labels copied to SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/labels/test


: 

In [ ]:
# Quick sanity check: ensure Ultralytics can see 4 channels via cached .npy files
sample_train = next((DUAL_ROOT / "images" / "train").glob("*.npy"), None)
if sample_train is None:
    raise FileNotFoundError("No cached dual tensors found under images/train. Run make_dual_split_npy first.")

arr = np.load(sample_train)
print(sample_train.name, arr.shape, arr.dtype)
assert arr.ndim == 3 and arr.shape[2] == 4, "Cached tensor must carry 4 channels"
print("✅ range channel preserved; Ultralytics BaseDataset will now load this .npy instead of 3-ch PNG")

In [ ]:
%%bash
set -euo pipefail
# ensure Ultralytics dataloader uses patched loader that respects .npy caches
python - <<'PY'
from ultralytics import YOLO
model = YOLO(r"SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/yolov9t_dual_4ch_fixed.pt")
model.info(verbose=True)
print("Model first conv weight shape:", model.model.model[0].conv.weight.shape)
PY

## The cell below is the main part about the dual network, currently just copying over one of the layer and adding as an 'alpha' channel for the range-normlaized images to get in. This can be done in many ways to improve but this is the 'naive' or first thought we had

In [23]:
# model = YOLO("yolov9t.pt")           # Step 2: load pretrained RGB weights
# model.model.eval()                   # get the underlying nn.Module graph

# first_conv = model.model.model[0]    # YOLOv9 stem (Conv → BN → SILU)
# old_conv = first_conv.conv
# new_conv = torch.nn.Conv2d(
#     in_channels=4,
#     out_channels=old_conv.out_channels,
#     kernel_size=old_conv.kernel_size,
#     stride=old_conv.stride,
#     padding=old_conv.padding,
#     bias=old_conv.bias is not None,
# )

# with torch.no_grad():
#     new_conv.weight[:, :3] = old_conv.weight            # copy RGB kernels
#     new_conv.weight[:, 3:] = old_conv.weight[:, :1] * 0 # or torch.randn_like(...)*1e-3
#     if old_conv.bias is not None:
#         new_conv.bias = old_conv.bias

# first_conv.conv = new_conv            # swap into the module tree
# model.model.model[0] = first_conv     # ensure the model graph is updated

# model.save("yolov9t_rgba.pt")

: 

In [6]:
ORIG_DATA_YAML = COMB_ROOT / "../" /"data.yaml"
DUAL_DATA_YAML = DUAL_ROOT / "data.yaml"

with open(ORIG_DATA_YAML, "r") as f:
    cfg = yaml.safe_load(f)

base = DUAL_ROOT

def make_rel(p):
    # p might be absolute or relative – we point to new dual root
    p = Path(p)
    return str((base / "images" / p.name).parent)  # keep split names

# If your original yaml used explicit paths, you can instead do:
# cfg["path"]  = str(DUAL_ROOT)
cfg["path"]  = str(DUAL_ROOT)
cfg["train"] = "images/train"
cfg["valid"]   = "images/valid"
cfg["test"]  = "images/test"
cfg["channels"] = 4          # tell YOLO this is 4-channel data with the RGB-Alpha

with open(DUAL_DATA_YAML, "w") as f:
    yaml.safe_dump(cfg, f)

print(DUAL_DATA_YAML.read_text())

channels: 4
names:
- pole
nc: 1
path: /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range
test: images/test
train: images/train
val: images/valid
valid: images/valid



: 

In [8]:
from pathlib import Path
import yaml

# 1. Define where the dual model yaml will live
DUAL_MODEL_YAML = DUAL_ROOT / "yolov9t_dual.yaml"
print("DUAL_MODEL_YAML:", DUAL_MODEL_YAML)

# # 2. Download official yolov9t.yaml from Ultralytics repo into that path
# !wget -O "{DUAL_MODEL_YAML}" https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/models/v9/yolov9t.yaml

# # 3. Load, patch ch and nc, and save back
# cfg["ch"] = 4          # 4 input channels (RGB + range)
# cfg = yaml.safe_load(DUAL_MODEL_YAML.read_text())
# cfg["nc"] = 1          # your number of classes

# # DUAL_MODEL_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False))
# print("Final yolov9t_dual.yaml content:\n")
# print(DUAL_MODEL_YAML.read_text())

DUAL_MODEL_YAML: /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/yolov9t_dual.yaml


: 

In [9]:
from ultralytics import YOLO
import torch

# Load model from the yaml (it will be 3-ch by default)
four_ch_model = YOLO(str(DUAL_MODEL_YAML))
net = four_ch_model.model

# Grab the first block and its Conv2d
first_block = net.model[0]
old_conv = first_block.conv          # this is Conv2d(3, 16, 3, 2, ...)

print("OLD conv shape:", old_conv.weight.shape)  # should show [16, 3, 3, 3]

# Create new 4-channel conv with same hyperparams
new_conv = torch.nn.Conv2d(
    in_channels=4,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    bias=(old_conv.bias is not None),
)

with torch.no_grad():
    # Copy existing 3-channel weights into first 3 channels
    new_conv.weight[:, :3, :, :] = old_conv.weight
    # Initialize 4th channel as zeros (or small noise if you prefer)
    new_conv.weight[:, 3:, :, :] = 0.0
    if old_conv.bias is not None:
        new_conv.bias.copy_(old_conv.bias)

# Swap into the model
first_block.conv = new_conv
net.model[0] = first_block
four_ch_model.model = net

print("NEW conv shape:", four_ch_model.model.model[0].conv.weight.shape)  # should be [16, 4, 3, 3]

# after modifying four_ch_model.model.model[0].conv to 4 channels
four_ch_model.save(str(DUAL_ROOT / "yolov9t_dual_4ch_fixed.pt"))
print("Saved:", DUAL_ROOT / "yolov9t_dual_4ch_fixed.pt")



OLD conv shape: torch.Size([16, 3, 3, 3])
NEW conv shape: torch.Size([16, 4, 3, 3])
Saved: /content/drive/MyDrive/SnowPole_Detection_Dataset/comb4-range-signal-reflec_and_range/yolov9t_dual_4ch_fixed.pt


: 

In [10]:
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
from pathlib import Path

class DualYOLODataset(Dataset):
    def __init__(self, images_dir: Path, labels_dir: Path, img_size=1024):
        self.images = sorted(images_dir.glob("*.png"))
        self.labels_dir = labels_dir
        self.img_size = img_size

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)  # keep 4 channels
        assert img is not None, f"Failed to read {img_path}"
        # img: H x W x 4, uint8

        # Resize to training size (keeping 4 channels)
        img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)

        # Convert to float32 and normalize 0–1
        img = img.astype(np.float32) / 255.0

        # HWC -> CHW
        img = np.transpose(img, (2, 0, 1))  # 4 x H x W

        # Load labels (YOLO txt) for that image
        stem = img_path.stem
        label_path = self.labels_dir / f"{stem}.txt"
        if label_path.exists():
            targets = np.loadtxt(str(label_path), ndmin=2)  # shape (N, 5) [cls, x, y, w, h]
        else:
            targets = np.zeros((0, 5), dtype=np.float32)

        sample = {
            "img": torch.from_numpy(img),        # 4 x H x W
            "cls": torch.from_numpy(targets[:, 0:1]) if targets.size else torch.zeros((0, 1), dtype=torch.float32),
            "bboxes": torch.from_numpy(targets[:, 1:5]) if targets.size else torch.zeros((0, 4), dtype=torch.float32),
            "im_file": str(img_path),
        }
        return sample


: 

In [11]:
!yolo train \
  model="{str(DUAL_ROOT / 'yolov9t_dual_4ch_fixed.pt')}" \
  data="{str(DUAL_DATA_YAML)}" \
  epochs=400 imgsz=1024 device=cpu batch=16 \
  name="dual_comb_rgb_plus_range_9t_4ch" \
  project="dual_comb_range_experiments"


In [60]:
from torch.utils.data import DataLoader
import torch

train_images = DUAL_ROOT / "images" / "train"
train_labels = DUAL_ROOT / "labels" / "train"

train_dataset = DualYOLODataset(train_images, train_labels, img_size=1024)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,
                          num_workers=0, collate_fn=lambda x: x)

device = torch.device("cpu")
four_ch_model.model.to(device)
four_ch_model.model.train()

optimizer = torch.optim.AdamW(four_ch_model.model.parameters(), lr=2e-3)
num_epochs = 400

#     for batch in train_loader:
#         imgs = torch.stack([b["img"] for b in batch], dim=0).to(device)  # [B, 4, H, W]

#         yolo_batch = {
#             "img": imgs,
#             "cls": [b["cls"].to(device) for b in batch],
#             "bboxes": [b["bboxes"].to(device) for b in batch],
#             "im_file": [b["im_file"] for b in batch],
#         }

#         optimizer.zero_grad()
#         loss, loss_items = four_ch_model.model(yolo_batch)
#         loss.backward()
#         optimizer.step()

#     print(f"Epoch {epoch+1}/{num_epochs} - loss: {loss.item():.4f}")
for epoch in range(num_epochs):
    for batch_i, batch in enumerate(train_loader):
        imgs = torch.stack([b["img"] for b in batch], dim=0).to(device)  # [B, 4, H, W]
        B = imgs.shape[0]

        cls_list = [b["cls"] for b in batch]       # each (Ni,1)
        box_list = [b["bboxes"] for b in batch]    # each (Ni,4)

        # Build batch_idx vector and concat targets like Ultralytics expects
        batch_idx = []
        targets_list = []
        for i, (cls_i, box_i) in enumerate(zip(cls_list, box_list)):
            if cls_i.numel() == 0:
                continue
            n = cls_i.shape[0]
            bi = torch.full((n, 1), i, dtype=torch.float32)
            targets_list.append(torch.cat([bi, cls_i, box_i], dim=1))  # (n,6)
            batch_idx.append(bi)

        if len(targets_list):
            targets = torch.cat(targets_list, dim=0)       # (N,6): [batch, cls, x,y,w,h]
            batch_idx_tensor = targets[:, 0].to(device)
            cls_tensor = targets[:, 1:2].to(device)
            bboxes_tensor = targets[:, 2:6].to(device)
        else:
            # no objects in batch
            batch_idx_tensor = torch.zeros((0,), dtype=torch.float32, device=device)
            cls_tensor = torch.zeros((0, 1), dtype=torch.float32, device=device)
            bboxes_tensor = torch.zeros((0, 4), dtype=torch.float32, device=device)

        yolo_batch = {
            "img": imgs,
            "batch_idx": batch_idx_tensor,
            "cls": cls_tensor,
            "bboxes": bboxes_tensor,
            "im_file": [b["im_file"] for b in batch],
        }

        optimizer.zero_grad()
        loss, loss_items = four_ch_model.model(yolo_batch)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{num_epochs} - loss: {loss.item():.4f}")


In [11]:
# # !yolo train \
# #   model="yolov9t_dual_4ch.pt" \
# #   data="{DUAL_DATA_YAML}" \
# #   epochs=400 imgsz=1024 device=cpu batch=16 name="dual_comb_rgb_plus_range_9t" project="dual_comb_range_experiments"


# # !yolo train \
# #   model="yolov9t_dual_4ch.pt" \
# #   data="{str(DUAL_DATA_YAML)}" \
# #   epochs=400 imgsz=1024 device=cpu batch=16 \
# #   name="dual_comb_rgb_plus_range_9t" \
# #   project="dual_comb_range_experiments"
# !yolo train \
#   model="{str(DUAL_MODEL_YAML)}" \
#   data="{str(DUAL_DATA_YAML)}" \
#   epochs=400 imgsz=1024 device=cpu batch=16 \
#   name="dual_comb_rgb_plus_range_9t4" \
#   project="dual_comb_range_experiments"


